# Day 6: Baseline Logistic Regression Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

import joblib

In [2]:
matchups = pd.read_csv("../data/processed/matchup_training_data.csv")
matchups.head()

,date,team_a,team_b,team_a_goals,team_b_goals,tournament,neutral,target,team_a_last_5_points_per_match,team_a_last_5_goals_for_per_match,...,team_b_last_5_win,team_b_last_5_draw,team_b_last_5_loss,last_5_points_per_match_diff,last_5_goals_for_per_match_diff,last_5_goals_against_per_match_diff,last_5_goal_difference_per_match_diff,last_5_win_diff,last_5_draw_diff,last_5_loss_diff
0,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,draw,0.000000,0.000000,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,win,0.500000,0.500000,...,0.0,0.0,0.0,0.500000,0.500000,2.000000,-1.500000,0.0,1.0,1.0
2,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,draw,1.333333,1.333333,...,5.0,0.0,0.0,-1.666667,-4.666667,0.933333,-5.600000,-4.0,1.0,1.0
3,2014-06-04,Abkhazia,South Ossetia,0.0,0.0,CONIFA World Football Cup,True,draw,1.600000,1.400000,...,1.0,0.0,2.0,0.600000,-5.266667,-0.800000,-4.466667,1.0,2.0,-1.0
4,2014-06-07,Abkhazia,Occitania,0.0,1.0,CONIFA World Football Cup,True,loss,1.800000,1.800000,...,2.0,3.0,0.0,0.000000,-0.400000,0.200000,-0.600000,0.0,0.0,0.0


In [3]:
matchups.shape
matchups.columns

Index(['date', 'team_a', 'team_b', 'team_a_goals', 'team_b_goals',
       'tournament', 'neutral', 'target', 'team_a_last_5_points_per_match',
       'team_a_last_5_goals_for_per_match',
       'team_a_last_5_goals_against_per_match',
       'team_a_last_5_goal_difference_per_match', 'team_a_last_5_win',
       'team_a_last_5_draw', 'team_a_last_5_loss',
       'team_b_last_5_points_per_match', 'team_b_last_5_goals_for_per_match',
       'team_b_last_5_goals_against_per_match',
       'team_b_last_5_goal_difference_per_match', 'team_b_last_5_win',
       'team_b_last_5_draw', 'team_b_last_5_loss',
       'last_5_points_per_match_diff', 'last_5_goals_for_per_match_diff',
       'last_5_goals_against_per_match_diff',
       'last_5_goal_difference_per_match_diff', 'last_5_win_diff',
       'last_5_draw_diff', 'last_5_loss_diff'],
      dtype='str')

In [44]:
matchups["target"].value_counts()
matchups["target"].value_counts(normalize=True)

target
win     0.489384
loss    0.282161
draw    0.228455
Name: proportion, dtype: float64

In [45]:
feature_columns = [
    "last_5_points_per_match_diff",
    "last_5_goals_for_per_match_diff",
    "last_5_goals_against_per_match_diff",
    "last_5_goal_difference_per_match_diff",
    "last_5_win_diff",
    "last_5_draw_diff",
    "last_5_loss_diff"
]

In [46]:
X = matchups[feature_columns]
y = matchups["target"]

In [47]:
X.head()

,last_5_points_per_match_diff,last_5_goals_for_per_match_diff,last_5_goals_against_per_match_diff,last_5_goal_difference_per_match_diff,last_5_win_diff,last_5_draw_diff,last_5_loss_diff
0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
2,-1.5,-1.000000,1.000000,-2.000000,-1.0,0.0,1.0
3,0.0,0.333333,-0.333333,0.666667,0.0,0.0,0.0
4,0.0,-0.250000,0.250000,-0.500000,0.0,0.0,0.0


In [48]:
y.head()

0    draw
1     win
2     win
3    draw
4     win
Name: target, dtype: str

In [49]:
matchups["date"] = pd.to_datetime(matchups["date"])
matchups = matchups.sort_values("date").reset_index(drop=True)
cutoff_date = "2018-01-01"
train_data = matchups[matchups["date"] < cutoff_date]
test_data = matchups[matchups["date"] >= cutoff_date]
train_data.shape, test_data.shape

((41299, 29), (8155, 29))

In [50]:
X_train = train_data[feature_columns]
y_train = train_data["target"]

X_test = test_data[feature_columns]
y_test = test_data["target"]

X_train.shape, X_test.shape

((41299, 7), (8155, 7))

In [51]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        class_weight="balanced" # added class_weight to address lack of draw predictions
    ))
])

In [52]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('logistic_regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['draw','loss','win']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['last_5_points_per_match_diff','last_5_goals_for_per_match_diff', 'last_5_goals_against_per_match_diff',...,'last_5_win_diff', 'last_5_draw_diff','last_5_loss_diff']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [53]:
probabilities = model.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilities,
    columns=model.classes_
)

prob_df.head()

,draw,loss,win
0,0.358000,0.323384,0.318617
1,0.343364,0.296796,0.359840
2,0.355059,0.323983,0.320958
3,0.356173,0.397979,0.245848
4,0.347512,0.344766,0.307722


In [54]:
def predict_with_draw_logic(
    row,
    draw_threshold=0.35,
    balance_threshold=0.08,
    max_side_threshold=0.33
):
    win_prob = row["win"]
    draw_prob = row["draw"]
    loss_prob = row["loss"]

    win_loss_balanced = abs(win_prob - loss_prob) <= balance_threshold
    neither_side_dominant = max(win_prob, loss_prob) <= max_side_threshold

    if (
        draw_prob >= draw_threshold
        and win_loss_balanced
        and neither_side_dominant
    ):
        return "draw"
    
    return row[["win", "loss"]].idxmax()

In [55]:
adjusted_pred = prob_df.apply(
    predict_with_draw_logic,
    axis=1,
    draw_threshold=0.35,
    balance_threshold=0.08,
    max_side_threshold=0.33
)

In [56]:
print("Adjusted predictions:")
print(pd.Series(adjusted_pred).value_counts())

print("Adjusted accuracy:", accuracy_score(y_test, adjusted_pred))
print(classification_report(y_test, adjusted_pred, zero_division=0))

Adjusted predictions:
win     4315
loss    3540
draw     300
Name: count, dtype: int64
Adjusted accuracy: 0.5091354996934396
              precision    recall  f1-score   support

        draw       0.29      0.04      0.08      1934
        loss       0.42      0.63      0.50      2359
         win       0.60      0.67      0.63      3862

    accuracy                           0.51      8155
   macro avg       0.43      0.45      0.40      8155
weighted avg       0.47      0.51      0.46      8155



In [57]:
y_pred = model.predict(X_test)
y_pred[:10]

array(['draw', 'win', 'draw', 'loss', 'draw', 'draw', 'win', 'loss',
       'loss', 'draw'], dtype=object)

In [58]:
comparison = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred
})

comparison.head(20)

,actual,predicted
0,win,draw
1,draw,win
2,draw,draw
3,draw,loss
4,loss,draw
5,loss,draw
6,loss,win
7,loss,loss
8,loss,loss
9,draw,draw


In [59]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.49061925199264256

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
labels = ["win", "draw", "loss"]

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm

In [ ]:
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
)

cm_df

In [ ]:
most_common_class = y_train.value_counts().idxmax()
most_common_class

In [ ]:
baseline_predictions = [most_common_class] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_accuracy

In [ ]:
print("Model accuracy:", accuracy)
print("Dumb baseline accuracy:", baseline_accuracy)

In [ ]:
probabilities = model.predict_proba(X_test)

In [ ]:
model.classes_

In [ ]:
probability_df = pd.DataFrame(
    probabilities,
    columns=model.classes_
)

probability_df.head()

In [ ]:
results_preview = test_data[
    ["date", "team_a", "team_b", "target"]
].reset_index(drop=True).copy()

results_preview = pd.concat(
    [results_preview, probability_df],
    axis=1
)

results_preview.head(20)

In [ ]:
joblib.dump(model, "../models/baseline_logistic_regression.pkl")

In [ ]:
results_preview.to_csv("../data/processed/baseline_predictions.csv", index=False)